# Part1 프로그래밍 언어 최종 종합 평가 — IoT 스마트 팩토리 센서 데이터 분석기

**이름:** ______________

- 위에서부터 셀을 **순서대로** 실행하세요 (Shift+Enter).
- `______` 로 표시된 빈칸을 채우거나, `# TODO` 가 있는 함수를 완성하세요.
- 각 셀 끝의 `[N단계] 결과` 출력이 `[N단계] 기대값` 과 같으면 그 단계는 완료입니다.
- 데이터 생성 셀과 확인용 `print` 코드는 수정하지 마세요.
- 모두 마치면 **파일 → 다운로드 → .ipynb** 로 저장해 제출합니다.

## 0단계. 데이터 파일 만들기 (수정하지 마세요)

In [ ]:
DATA = """2026-10-09 10:00:00,S1,23.5,60,0x00
2026-10-09 10:01:00,S2,31.2,55,0x01
2026-10-09 10:02:00,S1,24.1,62,0x00
2026-10-09 10:03:00,S3,29.8,70,0x00

2026-10-09 10:04:00,S2,33.0,50,0x01
2026-10-09 10:05:00,S1,ERROR,65,0x02
2026-10-09 10:06:00,S3,30.5,68,0x00
2026-10-09 10:07:00,S2,30.9,52,0x00
2026-10-09 10:08:00,S1,25.0,61,0x00
2026-10-09 10:09:00,S3,32.4,49,0x01
"""
with open("factory_sensor.txt", "w", encoding="utf-8") as f:
    f.write(DATA)
print("factory_sensor.txt 데이터 파일 생성 완료")

## 1단계. 한 줄 파싱 (빈칸 2개)

In [ ]:
def parse_log_line(line):
    """'2026-10-09 10:00:00,S1,23.5,60,0x00' -> ('S1', 23.5, 60, 0)"""
    parts = line.strip().______(",")          # 쉼표로 분리
    sensor_id = parts[1]
    temp = float(parts[2])                    # 문자열 -> 실수
    humid = int(parts[3])                     # 문자열 -> 정수
    flag = int(parts[4], ______)              # 16진수 문자열 -> 정수 (진법 지정)
    return sensor_id, temp, humid, flag

print("[1단계] 결과 :", parse_log_line("2026-10-09 10:00:00,S1,23.5,60,0x00"))
print("[1단계] 기대값: ('S1', 23.5, 60, 0)")

## 2단계. SensorRecord 클래스 정의 (TODO 클래스/메서드 구현)

In [ ]:
class SensorRecord:
    """센서 측정값 1건을 담는 클래스"""
    def __init__(self, sensor_id, temp, humid, flag):
        # TODO: 전달받은 4가지 센서 속성들을 인스턴스 변수로 할당하세요.
        pass

    def is_warning(self, limit=30.0):
        """온도가 limit 이상이거나 상태 플래그(flag)가 0이 아니면 True 반환"""
        # TODO: 온도가 기준치(limit) 이상이거나 상태 플래그에 오류 비트가 켜져 있으면 True, 정상 범위면 False를 반환하세요.
        pass

sample_rec = SensorRecord("S2", 31.2, 55, 1)
print("[2단계] 결과 :", sample_rec.sensor_id, sample_rec.is_warning(30.0))
print("[2단계] 기대값: S2 True")

## 3단계. 파일 읽기 및 예외 방어 (TODO 함수 구현)

In [ ]:
def load_factory_records(filepath):
    """파일을 읽어 SensorRecord 객체 리스트를 반환한다.
    빈 줄을 건너뛰고, 'ERROR' 등 파싱 에러가 발생한 행은 try-except로 건너뛴다."""
    records = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            clean_line = line.strip()
            if not clean_line:
                continue
            # TODO: 한 줄을 파싱하여 SensorRecord 객체를 생성해 records에 추가하되,
            #       결측치('ERROR') 등으로 파싱에 실패한 비정상 행은 프로그램 중단 없이 안전하게 건너뛰도록 예외를 방어하세요.
            pass
    return records

records = load_factory_records("factory_sensor.txt")
print("[3단계] 결과 :", len(records), records[0].sensor_id)
print("[3단계] 기대값: 9 S1")

## 4단계. 센서별 통계 집계 (빈칸 2개)

In [ ]:
def summarize_by_sensor(records):
    """센서별로 레코드를 묶고, 평균/최고/경고횟수를 딕셔너리로 반환"""
    groups = {}
    for r in records:
        if r.sensor_id ______ groups:         # 처음 보는 센서라면
            groups[r.sensor_id] = []
        groups[r.sensor_id].append(r)

    summary = {}
    for s_id, r_list in groups.items():
        temps = [r.temp for r in r_list]
        avg_temp = sum(temps) / len(temps)
        max_temp = ______(temps)             # 최댓값 함수 사용
        warn_count = sum(1 for r in r_list if r.is_warning(30.0))
        summary[s_id] = {
            "avg": round(avg_temp, 2),
            "max": round(max_temp, 2),
            "warn": warn_count,
            "count": len(temps)
        }
    return summary

summary = summarize_by_sensor(records)
print("[4단계] 결과 :", sorted(summary.keys()), summary["S1"]["count"], summary["S1"]["avg"])
print("[4단계] 기대값: ['S1', 'S2', 'S3'] 3 24.2")

## 5단계. Pandas 이상치 분석 (TODO 함수 구현)

In [ ]:
import pandas as pd

def find_high_temp_counts(records, limit=30.0):
    """SensorRecord 리스트로 DataFrame을 만들고,
    온도가 limit 이상인 데이터에 대해 센서별 건수(Series)를 반환"""
    data = [{"sensor_id": r.sensor_id, "temp": r.temp, "humid": r.humid} for r in records]
    df = pd.DataFrame(data)
    
    # TODO: DataFrame에서 기준치(limit) 이상의 고온 데이터만 필터링한 후,
    #       고온 경고가 발생한 센서별 누적 건수(빈도)를 집계하여 반환하세요.
    pass

alert_series = find_high_temp_counts(records, 30.0)
print("[5단계] 결과 :", dict(alert_series))
print("[5단계] 기대값: {'S2': 3, 'S3': 2}")

## 6단계. 모니터링 종합 리포트 출력 (빈칸 2개)

In [ ]:
def print_monitoring_report(summary):
    """센서별 통계 결과를 표 양식으로 정렬하여 콘솔에 출력"""
    print("=" * 48)
    print("센서ID    평균온도    최고온도    경고건수    측정수")
    print("-" * 48)
    for s_id in sorted(summary.keys()):
        info = summary[s_id]
        # f-string 형식 지정자로 정렬 출력 (소수점 둘째 자리 폭 8, 측정수는 info['count'])
        print(f"{s_id:<8} {info['avg']:>8.2f}    {info['max']:______}    {info['warn']:>6}    {info['count']:______}")
    print("=" * 48)

def format_summary_line(s_id, info):
    return f"{s_id}: avg={info['avg']:.2f}, max={info['max']:.2f}, warn={info['warn']}, count={info['count']}"

print("[6단계] 결과 :", format_summary_line("S1", summary["S1"]))
print("[6단계] 기대값: S1: avg=24.20, max=25.00, warn=0, count=3")
print("\n[모니터링 리포트 전체 표 출력]")
print_monitoring_report(summary)